# ContraTICO — Prompt Ablation Study

Full pipeline for the prompt ablation study across **3 strategies** × **3 configs** × **5 languages**:
1. **Source QA** — Each strategy × config combination
2. **BT QA** — Each strategy × config × language × perturbation
3. **Evaluation** — String Comparison + SBERT per strategy

| Parameter | Values |
|-----------|--------|
| Strategies | P1-fewshot, P2-cot, P3-concise |
| Configs | vanilla, atomic, semantic |
| Languages | es, fr, hi, tl, zh |
| Perturbations | alteration, expansion_impact, expansion_noimpact, intensifier, omission, spelling, synonym, word_order |

## 0. Environment Setup

In [ ]:
import os
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CACHE_DIR = '/content/drive/MyDrive/AskQE_Models_Cache'
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME'] = DRIVE_CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = os.path.join(DRIVE_CACHE_DIR, 'transformers')
    os.environ['SENTENCE_TRANSFORMERS_HOME'] = os.path.join(DRIVE_CACHE_DIR, 'sentence_transformers')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'torch', 'accelerate', 'nltk',
                'sentence-transformers', 'sacrebleu', 'textstat'], check=True)
print('Dependencies installed!')

In [ ]:
if IN_KAGGLE:
    PROJECT_ROOT = '/kaggle/working/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone',
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git',
                        PROJECT_ROOT], check=True)
elif IN_COLAB:
    PROJECT_ROOT = '/content/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone',
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git',
                        PROJECT_ROOT], check=True)
else:
    PROJECT_ROOT = os.getcwd()

print(f'Project root: {PROJECT_ROOT}')

## 1. Pre-download Qwen

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B-Instruct')
model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-3B-Instruct',
                                             torch_dtype=torch.bfloat16, device_map='auto')
del model, tokenizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('Qwen cached!')

## 2. Path Configuration

In [ ]:
ABLATION_DIR = f"{PROJECT_ROOT}/Qwen2.5-3B-Instruct/contratico/prompt-ablation"
CODE_DIR = f"{ABLATION_DIR}/code"
EVAL_DIR = f"{ABLATION_DIR}/evaluation"
BASELINE_DIR = f"{PROJECT_ROOT}/Qwen2.5-3B-Instruct/contratico/baseline"
CONTRATICO_DIR = f"{PROJECT_ROOT}/contratico"

STRATEGIES = ['P1-fewshot', 'P2-cot', 'P3-concise']
CONFIGS = ['vanilla', 'atomic', 'semantic']
LANGUAGES = ['es', 'fr', 'hi', 'tl', 'zh']

# Create output directories
for strategy in STRATEGIES:
    os.makedirs(f'{ABLATION_DIR}/QA/{strategy}', exist_ok=True)
    for config in CONFIGS:
        os.makedirs(f'{EVAL_DIR}/{strategy}/{config}/sbert', exist_ok=True)
        os.makedirs(f'{EVAL_DIR}/{strategy}/{config}/string-comparison', exist_ok=True)

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print(f'ABLATION_DIR: {ABLATION_DIR}')
print(f'BASELINE_DIR: {BASELINE_DIR}')

## 3. Source QA (All Strategies × All Configs)

In [ ]:
for strategy in STRATEGIES:
    for config in CONFIGS:
        cmd = [sys.executable, '-u', f'{CODE_DIR}/qa_ablation_contratico.py',
               '--strategy', strategy,
               '--mode', 'source',
               '--config', config,
               '--baseline_dir', BASELINE_DIR,
               '--output_dir', ABLATION_DIR]
        
        print(f'Source QA [{strategy}/{config}]...')
        subprocess.run(cmd, check=True)
        print(f'✓ Source QA [{strategy}/{config}] complete!')

print('\n✓ All Source QA runs complete!')

## 4. BT QA (All Strategies × All Configs × All Languages)

In [ ]:
for strategy in STRATEGIES:
    for config in CONFIGS:
        for lang in LANGUAGES:
            cmd = [sys.executable, '-u', f'{CODE_DIR}/qa_ablation_contratico.py',
                   '--strategy', strategy,
                   '--mode', 'bt',
                   '--config', config,
                   '--lang', lang,
                   '--baseline_dir', BASELINE_DIR,
                   '--contratico_dir', CONTRATICO_DIR,
                   '--output_dir', ABLATION_DIR]
            
            print(f'BT QA [{strategy}/{config}/{lang}]...')
            subprocess.run(cmd, check=True)
            print(f'✓ BT QA [{strategy}/{config}/{lang}] complete!')

print('\n✓ All BT QA runs complete!')

## 5. Evaluation (All Strategies)

Runs string comparison and SBERT for each strategy.

In [ ]:
eval_script = f'{CODE_DIR}/evaluation_contratico.py'

for strategy in STRATEGIES:
    cmd = [sys.executable, '-u', eval_script,
           '--strategy', strategy,
           '--ablation_dir', ABLATION_DIR,
           '--baseline_dir', BASELINE_DIR]
    
    print(f'\nEvaluation [{strategy}]...')
    subprocess.run(cmd, check=True)
    print(f'✓ Evaluation [{strategy}] complete!')

print('\n✓ All Evaluations complete!')

## Summary

Pipeline complete! Output structure:
```
prompt-ablation/
├── QA/{P1-fewshot,P2-cot,P3-concise}/
└── evaluation/{P1-fewshot,P2-cot,P3-concise}/
    └── {vanilla,atomic,semantic}/
        ├── sbert/
        └── string-comparison/
```